# Module 05 — Notebook 4 Solutions: Eval Plots Mini-Project

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
%matplotlib inline
sns.set_theme(style="whitegrid")
df = pd.read_csv(Path("../../../data/synthetic/evaluation_results.csv"))
with open(Path("../../../data/synthetic/model_outputs.json")) as f:
    outputs_df = pd.DataFrame(json.load(f))
outputs_df["response_length"] = outputs_df["response"].str.len()
OUTPUT_DIR = Path("../../../output/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Exercise 1 Solution

In [ ]:
per_model    = df.groupby("model")["score"].mean().sort_values(ascending=False)
per_task     = df.groupby("task")["score"].mean().sort_values()
scorecard    = df.pivot_table(index="model", columns="task", values="score")
overall_mean = round(float(df["score"].mean()), 4)

In [ ]:
check_type(per_model, pd.Series, "per_model is a Series")
check_type(per_task, pd.Series, "per_task is a Series")
check_type(scorecard, pd.DataFrame, "scorecard is a DataFrame")
check_equal(scorecard.shape, (4, 5), "scorecard is 4×5")
check_approx(overall_mean, 0.8225, 1e-4, "overall_mean")

## Exercise 2 Solution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].bar(per_model.index, per_model.values, color="steelblue", alpha=0.85)
axes[0, 0].set_title("Mean Score by Model")
axes[0, 0].tick_params(axis="x", rotation=15)

axes[0, 1].barh(per_task.index, per_task.values, color="steelblue", alpha=0.85)
axes[0, 1].set_title("Mean Score by Task")

sns.boxplot(data=df, x="model", y="score", palette="muted", ax=axes[1, 0])
axes[1, 0].set_title("Score Distribution by Model")
axes[1, 0].tick_params(axis="x", rotation=15)

sns.heatmap(scorecard, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=0.5, ax=axes[1, 1])
axes[1, 1].set_title("Scorecard")
axes[1, 1].tick_params(axis="x", rotation=30)

fig.suptitle("Evaluation Summary", fontsize=14)
plt.tight_layout()

summary_path = OUTPUT_DIR / "summary_4panel.png"
fig.savefig(summary_path, dpi=150, bbox_inches="tight")
print(f"Saved: {summary_path}")
plt.show()

In [ ]:
check_type(summary_path, Path, "summary_path is a Path")
check_equal(summary_path.exists(), True, "summary_4panel.png was saved")

## Exercise 3 Solution

In [ ]:
flag_rates = outputs_df.groupby("model")["flagged"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["crimson" if r > 0.5 else "steelblue" for r in flag_rates.values]
ax.bar(flag_rates.index, flag_rates.values, color=colors, alpha=0.85)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="50% threshold")
ax.set_title("Flag Rate by Model")
ax.set_ylabel("Flag Rate")
ax.set_ylim(0, 1.0)
ax.legend()
plt.tight_layout()

flag_path = OUTPUT_DIR / "flag_rates.png"
fig.savefig(flag_path, dpi=150, bbox_inches="tight")
print(f"Saved: {flag_path}")
plt.show()

In [ ]:
check_type(flag_rates, pd.Series, "flag_rates is a Series")
check_approx(float(flag_rates["model-a-v1"]), 0.0, 1e-6, "model-a-v1 flag rate is 0")
check_type(flag_path, Path, "flag_path is a Path")
check_equal(flag_path.exists(), True, "flag_rates.png was saved")

## Exercise 4 Solution

In [ ]:
saved_plots = sorted([p.name for p in OUTPUT_DIR.glob("*.png")])
print("Saved plots:", saved_plots)

In [ ]:
check_type(saved_plots, list, "saved_plots is a list")
check_contains(saved_plots, "flag_rates.png", "flag_rates.png present")
check_contains(saved_plots, "summary_4panel.png", "summary_4panel.png present")
check_contains(saved_plots, "scorecard_heatmap.png", "scorecard_heatmap.png present")